# 04 Pre-trained Features and Classifiers

**Owner:** ______  |  **Status:** Not started

**Goal:** extract embeddings from a pre-trained CNN (CPU only), visualize them, and classify with two methods.

**Rubric items this feeds:** Feature Extraction (25), Classification (25), Generalizability (10), and the inputs to Efficiency vs. Accuracy (10).

**Depends on:** notebook 01 (cropped signs and split CSVs). Until the full split lands, prototype on the ~20 starter crops per class.

**Hand-off checklist (tick before telling the team you're done):**
- [ ] features cached with `save_features(..., "pretrained", split)` for train, val and test
- [ ] both classifiers logged with `record_result(...)`
- [ ] explanation cells filled in (sections 7 and 11), since these become the report

## 1. Setup and load the split

In [ ]:
import sys
sys.path.append("..")  # run from notebooks/, so ".." is the repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import (
    SEED, IMAGE_SIZE, CLASSES, DATA_RAW, DATA_PROCESSED, DATA_SPLITS,
    load_split, load_image, load_split_images, save_features, load_features,
    timed, evaluate, plot_confusion_matrix, record_result,
)

np.random.seed(SEED)

In [ ]:
train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")
X_train_img, y_train = load_split_images("train")
X_val_img, y_val = load_split_images("val")
X_test_img, y_test = load_split_images("test")
print(X_train_img.shape, X_val_img.shape, X_test_img.shape)

## 2. Define the pre-trained feature extractor

- Use a torchvision model pre-trained on ImageNet (`resnet18` is a good CPU-friendly start; `resnet50` is heavier). Drop the final classification layer and use the pooled embedding.
- Resize to the model's expected input and apply ImageNet normalization.
- Run in `model.eval()` with `torch.no_grad()` and batch the images.
- Embeddings are extracted **once** and cached; do not re-run the CNN inside the classifier loop.

In [ ]:
import torch
import torchvision
from torchvision import transforms

device = torch.device("cpu")

def build_extractor():
    # TODO: load a pre-trained resnet (for example torchvision.models.resnet18 with weights),
    #       replace model.fc with torch.nn.Identity(), call .eval()
    pass

preprocess = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_pretrained(model, images, batch_size=32):
    # TODO: run batches through the model and return an (N, D) numpy array
    pass

## 3. Illustrate the feature

TODO: show something that makes the learned feature concrete, for example early-layer feature maps for a few signs, or for one query sign the 5 nearest neighbors in embedding space.

In [ ]:
# TODO: feature maps or nearest-neighbor illustration

## 4. Extract and cache the features

Extraction time counts toward the cost in the efficiency comparison, so time it. Save with `save_features` so rows line up with the split CSVs.

In [ ]:
FEATURE_NAME = "pretrained"

# TODO: extract features for each split, timing each call with timed(...),
#       then save_features(X, FEATURE_NAME, split_name)
feature_time_s = None  # total extraction seconds across train + val + test

## 5. PCA

TODO: 2D PCA scatter colored by class, plus a plot of cumulative explained variance. Describe what the PCA shows about how separable the classes are.

In [ ]:
from sklearn.decomposition import PCA
# TODO

## 6. t-SNE

TODO: t-SNE scatter colored by class (try a couple of perplexity values on a subset if it is slow). Compare with the PCA plot.

In [ ]:
from sklearn.manifold import TSNE
# TODO

## 7. Why this feature

TODO: explain why a pre-trained network is a good fit (it can read symbols and text the hand-designed feature misses), which model and layer you used and why, and the cost on CPU.

## 8. Classifier A: Logistic Regression

- Standardize features (`StandardScaler`, **fit on train only**).
- Use `class_weight="balanced"` because `stop` has far fewer examples than `speedlimit`.
- Search hyperparameters (for example `C`) by training on train and scoring on **validation**. Never use the test set here.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
# TODO: load cached features, scale (fit on train only), grid search C on validation

## 9. Classifier B: SVM

- Same scaling and class weighting as above.
- Search `C` and the kernel (linear vs. RBF, plus `gamma` for RBF) using validation only.
- Any two methods from class are fine (for example logistic regression, SVM, perceptron). Change these two if you prefer, but keep the **same two classifiers across notebooks 02-04** so the six combos are comparable.

In [ ]:
from sklearn.svm import SVC
# TODO: grid search C / kernel / gamma on validation

## 10. Final evaluation (run once)

With hyperparameters chosen, fit each classifier with its best settings and evaluate on the **test** set one time.

For each classifier:
- time training and time inference (use `timed`)
- compute accuracy and the per-class report with `evaluate(...)`
- plot the confusion matrix with `plot_confusion_matrix(...)`
- log it: `record_result(FEATURE_NAME, "<classifier>", val_acc, test_acc, train_time_s, infer_time_s, best_params, per_class_df, feature_time_s)`

In [ ]:
# TODO: Classifier A on the test set + record_result(...)

In [ ]:
# TODO: Classifier B on the test set + record_result(...)

## 11. Per-class results and limitations

TODO (write in full sentences, this feeds the report):
- Which classes did each classifier get right or wrong, and why? (look at the confusion matrices)
- How did the class imbalance show up in the results?
- What are the limits of this feature + classifier pairing?
- How much did tuning on validation help, and is there a gap between validation and test?